In [15]:
import numpy as np
import seaborn as sns
import pandas as pd

In [16]:
train = pd.read_csv('train_imputed.csv')
test = pd.read_csv('test_imputed.csv')
y = train[["SalePrice"]]
train.drop(columns=["SalePrice"],inplace=True)

In [17]:
corr = train.corr().abs()

# Upper triangle mask (to avoid duplicate checks)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Identify columns to drop
to_drop = [column for column in upper.columns if any(upper[column] > 0.75)]
train.drop(columns=to_drop, inplace=True)

In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X = train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
reg = LinearRegression().fit(X_train,y_train)
y_pred = reg.predict(X_test)
mean_squared_error(y_test, y_pred)

1203364704.1069825

In [19]:
reg.score(X_test,y_test)

0.8431142729002288

In [20]:
import statsmodels.api as sm

def stepwise_aic_selection(X, y, 
                           initial_features=None, 
                           threshold_in=0.01, 
                           threshold_out=0.05, 
                           verbose=True):
    """
    Perform a forward-backward feature selection based on AIC.
    
    Parameters:
    ----------
    X : pandas.DataFrame
        Candidate features.
    y : pandas.Series
        Target variable.
    initial_features : list
        Starting list of features (default: none).
    threshold_in : float
        p-value threshold for adding a feature.
    threshold_out : float
        p-value threshold for removing a feature.
    verbose : bool
        If True, prints progress.
    
    Returns:
    -------
    list
        Selected features.
    """
    included = list(initial_features) if initial_features else []
    while True:
        changed = False

        # Forward step
        excluded = list(set(X.columns) - set(included))
        new_pvalues = pd.Series(index=excluded, dtype=float)
        for new_col in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_col]])).fit()
            new_pvalues[new_col] = model.pvalues[new_col]
        best_pval = new_pvalues.min() if not new_pvalues.empty else None
        if best_pval is not None and best_pval < threshold_in:
            best_feature = new_pvalues.idxmin()
            included.append(best_feature)
            changed = True
            if verbose:
                print(f'Add {best_feature:>20} with p-value {best_pval:.4f}')

        # Backward step
        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        # use all coefs except intercept
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            changed = True
            if verbose:
                print(f'Remove {worst_feature:>20} with p-value {worst_pval:.4f}')

        if not changed:
            break

    final_model = sm.OLS(y, sm.add_constant(X[included])).fit()
    if verbose:
        print("\nFinal Model AIC:", final_model.aic)
        print("Selected features:", included)
    return included, final_model

In [21]:
stepwise_aic_selection(X_train,y_train)

Add          OverallQual with p-value 0.0000
Add            GrLivArea with p-value 0.0000
Add           BsmtFinSF1 with p-value 0.0000
Add           GarageCars with p-value 0.0000
Add            ExterQual with p-value 0.0000
Add      MSSubClass_freq with p-value 0.0000
Add         BsmtExposure with p-value 0.0000
Add          KitchenQual with p-value 0.0000
Add    Neighborhood_freq with p-value 0.0000
Add      Condition1_freq with p-value 0.0001
Add           Fireplaces with p-value 0.0001
Add           Functional with p-value 0.0003
Add              LotArea with p-value 0.0019
Add       RoofStyle_freq with p-value 0.0026
Add     No. of bathrooms with p-value 0.0010
Add        BldgType_freq with p-value 0.0003
Add           HouseStyle with p-value 0.0033
Add           MasVnrArea with p-value 0.0026
Add          OverallCond with p-value 0.0065
Add             BsmtQual with p-value 0.0028
Add             BsmtCond with p-value 0.0001

Final Model AIC: 27600.485761850225
Selected features:

(['OverallQual',
  'GrLivArea',
  'BsmtFinSF1',
  'GarageCars',
  'ExterQual',
  'MSSubClass_freq',
  'BsmtExposure',
  'KitchenQual',
  'Neighborhood_freq',
  'Condition1_freq',
  'Fireplaces',
  'Functional',
  'LotArea',
  'RoofStyle_freq',
  'No. of bathrooms',
  'BldgType_freq',
  'HouseStyle',
  'MasVnrArea',
  'OverallCond',
  'BsmtQual',
  'BsmtCond'],
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x20c2d069e10>)

In [22]:
train = train[['OverallQual', 'GrLivArea', 'BsmtFinSF1', 'GarageCars', 'ExterQual', 'MSSubClass_freq', 'BsmtExposure', 'KitchenQual', 'Neighborhood_freq', 'Condition1_freq', 'Fireplaces', 'Functional', 'LotArea', 'RoofStyle_freq', 'No. of bathrooms', 'BldgType_freq', 'HouseStyle', 'MasVnrArea', 'OverallCond', 'BsmtQual', 'BsmtCond']]
test = test[['OverallQual', 'GrLivArea', 'BsmtFinSF1', 'GarageCars', 'ExterQual', 'MSSubClass_freq', 'BsmtExposure', 'KitchenQual', 'Neighborhood_freq', 'Condition1_freq', 'Fireplaces', 'Functional', 'LotArea', 'RoofStyle_freq', 'No. of bathrooms', 'BldgType_freq', 'HouseStyle', 'MasVnrArea', 'OverallCond', 'BsmtQual', 'BsmtCond']]


In [23]:
X = train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
reg = LinearRegression().fit(X_train,y_train)
y_pred = reg.predict(X_test)
mean_squared_error(y_test, y_pred)

1173092144.051245

In [24]:
reg.score(X_test,y_test)

0.8470609838011774

In [25]:

from sklearn.tree import DecisionTreeRegressor
X = train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
reg = DecisionTreeRegressor().fit(X_train,y_train)
y_pred = reg.predict(X_test)
mean_squared_error(y_test, y_pred)

1275594328.8664384

In [26]:
import xgboost as xgb

X = train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(objective='reg:squarederror',
                         n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mean_squared_error(y_test, y_pred)


702633015.2469382

In [27]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
pred = reg.predict(test)

In [33]:
pd.Series(pred).to_csv("test1.csv")